Author: Konstantin Malanchev and the LINCC Frameworks team

Last updated: November 2026

# Homework 1b: Palomar 12 color-magnitude diagram

**Task:** plot a color-magnitude diagram of the globular cluster Palomar 12.

**Plan:**
1. Select a small cone around the cluster with `ConeSearch`
2. Select point sources with good photometry
3. Plot a color-magnitude diagram

If you'd like to solve it yourself, `Notebook_1a_Intro` may help.

> [!NOTE]
> This notebook uses Rubin DP2 data, so it can be run by Rubin data rights holders (DRHs) only, e.g. on the [Rubin Science Platform](https://data.lsst.cloud) with the latest weekly image.
> Since we would like to share the notebook and its outputs with everyone, we never print or plot individual rows of the DP2 catalog: all the outputs are either catalog metadata or aggregated statistics.

## Solution

In [ ]:
# lsdb is pre-installed on RSP, uncomment the line below if you are on another platform
# %pip install -qq lsdb

In [ ]:
# Standard library imports
import logging
import os
import warnings

# Third-party imports
import dask
import lsdb
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dask.distributed import Client
from lsdb import ConeSearch
from matplotlib import rcParams

# Jupyter magic commands
%matplotlib inline

# Configuration
warnings.simplefilter("ignore")
logging.getLogger("distributed").setLevel(logging.WARNING)
rcParams["savefig.dpi"] = 150
rcParams["font.size"] = 14
plt.rc("font", family="serif")
mpl.rcParams["axes.linewidth"] = 1.5
dask.config.set({"dataframe.convert-string": False})

## Setup Dask cluster

We use the same Dask setup as in `Notebook_1a_Intro`: sixteen workers, one per CPU of a "Large" RSP container, with one thread and 2 GB of memory each, which is enough for this low-memory pipeline, and `/deleted-sundays/` for spilling data to disk.

In [ ]:
client = Client(
    n_workers=16,
    threads_per_worker=1,
    local_directory=f"/deleted-sundays/{os.environ.get('USER', 'dask_scratch')}",
)

# Print the dashboard link and port
print(f"Dask is running at: {client.dashboard_link}")

### Select the cluster

We select a 0.5-degree cone around the cluster center with `lsdb.ConeSearch`. The coordinates are from the [Harris (1996, 2010 edition)](https://physics.mcmaster.ca/~harris/mwgc.dat) catalog of Milky Way globular clusters.

In [ ]:
# 0.5-degree cone around Palomar 12
cone_search = ConeSearch(ra=326.66, dec=-21.25, radius_arcsec=0.5 * 3600)

# DP2 Object collection on RSP, check https://lsdb.io/dp2 for the paths on other platforms
object_cluster = lsdb.open_catalog(
    "/rubin/lsdb_data/dp2/object_collection",
    # coordinates and _healpix_29 are always loaded
    columns=["objectId", "refExtendedness", "r_psfMag", "r_psfMagErr", "z_psfMag", "z_psfMagErr"],
    search_filter=cone_search,
)
object_cluster

### Select point sources

As in `Notebook_1a_Intro`, we select point-like objects, but with a less strict cut on PSF magnitude errors: smaller than 0.1 mag in r and z bands.

In [ ]:
stars = object_cluster.query("refExtendedness == 0 and r_psfMagErr < 0.1 and z_psfMagErr < 0.1")
stars

### Color-magnitude diagram

We plot r magnitude vs r−z color as a 2D histogram, also known as a Hess diagram. As in `Notebook_1a_Intro`, we compute partial histograms with `map_partitions()` and sum them, so only bin counts leave the workers.

In [ ]:
RZ_BINS = np.linspace(-0.5, 2.5, 61)
R_BINS = np.linspace(16.0, 25.0, 91)


def color_magnitude_hist(df):
    """Partial 2D histogram of r-z vs r for a single partition."""
    counts, _, _ = np.histogram2d(df["r_psfMag"] - df["z_psfMag"], df["r_psfMag"], bins=[RZ_BINS, R_BINS])
    rz_idx, r_idx = np.nonzero(counts)
    return pd.DataFrame({"rz_bin": rz_idx, "r_bin": r_idx, "count": counts[rz_idx, r_idx].astype(np.int64)})


partial_hists = stars.map_partitions(color_magnitude_hist, meta={"rz_bin": int, "r_bin": int, "count": int})

In [ ]:
partial_hists_df = partial_hists.compute()

hist = np.zeros((len(RZ_BINS) - 1, len(R_BINS) - 1), dtype=np.int64)
np.add.at(hist, (partial_hists_df["rz_bin"], partial_hists_df["r_bin"]), partial_hists_df["count"])

# A bin with a single object exposes its photometry up to the bin size, so we remove such bins
hist[hist == 1] = 0
print(f"Total number in the histogram: {hist.sum():,}")

In [ ]:
# Empty bins are NaN, so they are not colored
hist_plot = np.where(hist > 0, hist, np.nan)

fig, ax = plt.subplots(figsize=(8, 10))
mesh = ax.pcolormesh(RZ_BINS, R_BINS, hist_plot.T, cmap="viridis")
fig.colorbar(mesh, ax=ax, label="Number")
ax.invert_yaxis()
ax.set_xlabel("$r - z$ [mag]")
ax.set_ylabel("$r$ [mag]")
ax.set_title("Rubin DP2 point sources around Palomar 12")
plt.tight_layout()
plt.show()

In [ ]:
client.close()